# Notebook 2 – Data Type Handling

## 1. Identifying data types
**Concept:** Data types tell Python how values should be interpreted. Numerical, categorical, Boolean, date/time, and string columns may need different handling.

**Example:** `TotalCharges` may be stored as text even though it represents a number.

In [1]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "tenure": [12, 24, 8],
    "TotalCharges": ["846.0", "2040.0", " "],
    "JoinDate": ["2024-01-05", "invalid", "2024-03-12"],
    "Contract": ["Month-to-month", "One year", "Two year"],
    "IsActive": [True, False, True]
})
print(df.dtypes)
display(df.head())

tenure          int64
TotalCharges      str
JoinDate          str
Contract          str
IsActive         bool
dtype: object


,tenure,TotalCharges,JoinDate,Contract,IsActive
0,12,846.0,2024-01-05,Month-to-month,True
1,24,2040.0,invalid,One year,False
2,8,,2024-03-12,Two year,True


**Output explanation:** `dtypes` shows how each column is currently stored. A string column can contain numbers or dates that need conversion.

## 2. Numerical, categorical, Boolean, date/time, and string data
**Concept:** Numerical data supports arithmetic; categorical data represents groups; Boolean data represents true/false; date/time data represents time points; strings store text.

**Example:** `tenure` is numerical, `Contract` is categorical, `IsActive` is Boolean, and `JoinDate` should be datetime.

In [2]:
print("Numerical:", df.select_dtypes(include=np.number).columns.tolist())
print("Boolean:", df.select_dtypes(include="bool").columns.tolist())
print("String/object:", df.select_dtypes(include=["object", "string"]).columns.tolist())

Numerical: ['tenure']
Boolean: ['IsActive']
String/object: ['TotalCharges', 'JoinDate', 'Contract']


**Output explanation:** The output groups columns by their current storage type. Object/string columns should be inspected rather than assumed to be ordinary text.

## 3. Converting data types
**Concept:** Convert columns to types that match their meaning. Use `astype()` for direct conversions, `to_numeric()` for numeric parsing, and `to_datetime()` for dates.

**Examples:** String → Numeric, String → Date, Numeric → Categorical, Integer → Float.

In [3]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")
df["JoinDate"] = pd.to_datetime(df["JoinDate"], errors="coerce")
df["tenure_float"] = df["tenure"].astype(float)
df["tenure_category"] = df["tenure"].astype("category")
print(df.dtypes)
display(df)

tenure                      int64
TotalCharges              float64
JoinDate           datetime64[us]
Contract                      str
IsActive                     bool
tenure_float              float64
tenure_category          category
dtype: object


,tenure,TotalCharges,JoinDate,Contract,IsActive,tenure_float,tenure_category
0,12,846.0,2024-01-05,Month-to-month,True,12.0,12
1,24,2040.0,NaT,One year,False,24.0,24
2,8,NaN,2024-03-12,Two year,True,8.0,8


**Output explanation:** Valid strings become numbers/dates. Invalid or blank entries become missing (`NaN`/`NaT`) because `errors="coerce"` was selected. The category conversion treats tenure values as categories, which is only appropriate when the modeling meaning supports that choice.

## 4. Handling invalid conversions and detecting incorrect types
**Concept:** Invalid conversions should be counted and investigated. Do not silently treat every conversion failure as harmless.

In [4]:
raw_charges = pd.Series(["100.5", "unknown", "", "250"], name="charges")
converted = pd.to_numeric(raw_charges, errors="coerce")
invalid_mask = converted.isna() & raw_charges.astype("string").str.strip().ne("")
print("Original:", raw_charges.tolist())
print("Converted:", converted.tolist())
print("Nonblank invalid values:", raw_charges[invalid_mask].tolist())

Original: ['100.5', 'unknown', '', '250']
Converted: [100.5, nan, nan, 250.0]
Nonblank invalid values: ['unknown']


**Output explanation:** `unknown` fails numeric conversion, while the blank is also missing after conversion. The mask isolates nonblank invalid values for review.